# E2E Austria AML run

> **Run [`00_build_images.ipynb`](./00_build_images.ipynb) first.** This notebook assumes
> `fsd-aml-env` and `fsd-infer-sklearn` are already registered **and finished building** in ACR.
> It does not take their versions on trust: the config cell declares the same two
> `fsd.image.ImageDefinition`s and asks the image registry which versions they became, and
> stops if either one had to be built (spec 56 D7 — there is nothing to paste any more).

Init

```bash
python3.11 -m venv .venv
source .venv/bin/activate
pip install -e ".[dev,aml,mpc,azure,grid]"
```

## What lives where

```
notebooks/
  demo_model/       YOUR model only: my_adapter.py + rf.joblib
  demo_bundle/      written by bundle.save below
  demo_verify_adapter/<RUN>/  written by fsd.verify_adapter below: one real cube,
                    output.tif, grids.geojson, _result.json -- open the tif in QGIS
  demo_training_data/  written by create_training_data below
  (the model registry is NOT here, and NOT under your run root either -- it lives on blob
   at the durable path `model_registry` in ~/.config/fsd/config.toml (spec 55 D2), because
   models outlive the runs that made them: one folder per model name, one immutable v<N>
   per deployed bundle, plus _aliases.json)
```

`bundle.save` reads only `demo_model/`. There are no image build contexts in this repo since
spec 56 — `fsd.image.ImageDefinition` renders the Dockerfile and builds the wheel into a temp
directory at build time, and `verify_image` catches a stale image by reading the image
registry's record of what went into it rather than a wheel left behind on disk.
## The shape of the whole thing

```
1. create training data   fsd.create_training_data                    -> section 1
2. create features        fsd.bands.modify + apply_features           -> section 2.1
3. train the model        your own sklearn code, not fsd's            -> section 2.2
4. deploy the model       five steps, each one a gate                 -> section 3
5. run inference          fsd.download, then fsd.run_inference        -> section 4
```

Two steps hide inside those five, and both are easy to forget:

- **Step 5 begins with a download.** `run_inference` never fetches imagery — it reads a catalog
  and refuses to go to the source. That is deliberate: a fan-out of 299 nodes must not each
  decide to hit CDSE. So inference over a new ROI means `fsd.download` first, then
  `fsd.run_inference`. Step 1 has a download too, but it is folded inside
  `create_training_data(download=True)`, which is why you do not call it yourself there.
- **Before any of it: the images.** `00_build_images.ipynb` registers `fsd-aml-env` (the pipeline
  legs) and `fsd-infer-sklearn` (inference). Once per dependency *family*, never per model —
  retraining does not touch them. Both notebooks call `fsd.aml.ensure_environment` against the
  same image registry, so whichever you run second reuses what the first one built.


## 0. Setup

In [ ]:
import datetime, geopandas as gpd, fsd

In [ ]:
# Your Azure settings live in ~/.config/fsd/config.toml (outside this repo) and are read by
# fsd.config.load(). Run `fsd init` once to fill them in; see specs 54 and 55.
import os
import pathlib

import fsd.aml
from fsd.image import ImageDefinition

REPO = pathlib.Path.cwd().parent
assert (REPO / "pyproject.toml").exists(), f"expected an fsd checkout at {REPO}"
NOTEBOOKS = REPO / "notebooks"

cfg = fsd.config.load()

# AZ_ROOT is deliberately NOT config (spec 55 D1): a storage root is per-project and per-run,
# chosen by whoever runs this, so fsd takes it as an argument rather than storing it. This
# notebook is committed to a public repo and may not hold a literal storage URL, so it reads
# one from the environment -- export it before starting the kernel.
AZ_ROOT = os.environ.get("AZ_ROOT")
assert AZ_ROOT, "set AZ_ROOT (export AZ_ROOT=abfss://…) — spec 55 D1: root is not config"

# Both images come from 00_build_images.ipynb -- and since spec 56 there is nothing to paste.
# The SAME two definitions, digested the same way, ask the SAME registry, so this notebook
# RESOLVES the versions that notebook registered instead of being told them (D7). A stale
# paste used to point at a real image that was simply the wrong one; a digest cannot.
#
# The inference image is generic per DEPENDENCY FAMILY (sklearn), never per model: rebuild it
# when the fsd source or the DEPS change, NOT when you retrain or edit the adapter -- both of
# those ride inside the bundle, so one image serves every sklearn model you have.
IMAGE_REGISTRY = cfg.image_registry
assert IMAGE_REGISTRY, "set image_registry: `fsd init --set image_registry=abfss://…/image_registry`"

# Identical to 00_build_images.ipynb's declarations -- that is the point. `fsd="path:..."`
# hashes the CONTENT of a wheel built from THIS checkout (spec 56 D5), so an uncommitted edit
# is a different image with no separate git-dirty check to remember. It costs one `pip wheel`
# per definition, which is why this cell is the slow one.
BASE = ImageDefinition(
    name="fsd-aml-env",
    fsd=f"path:{REPO}",
    extras=("azure", "mpc"),      # see 00_build_images.ipynb for why not [aml] or [grid]
)
INFER = BASE.derive(name="fsd-infer-sklearn", extra_pip=("scikit-learn", "joblib"))

_env = fsd.aml.ensure_environment(
    BASE, registry=IMAGE_REGISTRY,
    resource_group=cfg.resource_group, workspace=cfg.workspace,
    storage="azure",   # the image registry is on blob, same as the model registry below
)
_infer = fsd.aml.ensure_environment(
    INFER, registry=IMAGE_REGISTRY,
    resource_group=cfg.resource_group, workspace=cfg.workspace,
    storage="azure",   # the image registry is on blob, same as the model registry below
)

# reused=False means this cell just STARTED an ACR build -- 10-20 minutes, and nothing below
# can run against an environment whose image has not finished. Fail here rather than at the
# first job submission twenty cells later.
for _r in (_env, _infer):
    assert _r.reused, (
        f"{_r.ref} was just built -- open {_r.build_url} , wait for `Build status: Succeeded`, "
        "then re-run this cell. It will say `reusing` and carry on."
    )

AZ_ENV_NAME, AZ_ENV_VERSION = _env.name, _env.version
AZ_INFER_ENV_NAME, AZ_INFER_ENV_VERSION = _infer.name, _infer.version
print(f"images: {_env.ref} / {_infer.ref}   (resolved from the registry, not pasted)")

# The model registry: a NAME for a trained model, so the inference cell can say
# "crop-rf@demo-eurocrops-at" instead of a path to a folder. Promoting a retrained model then
# becomes one alias reassignment instead of an edit to the call.
#
# It lives on BLOB, not on this laptop. That is the entire point of a registry: the name
# resolves from anywhere -- your machine, a colleague's, a cluster node -- rather than pointing
# at a folder only you have. (This needed a local folder until 2026-08-25; see section 3.5.)
#
# Note it does NOT hang off ROOT. ROOT is per-run; a registry that disappeared with the run
# would be a folder with extra steps. Models outlive the runs that made them -- which is
# exactly why the registry IS config (spec 55 D2) while the run root is not.
REGISTRY   = cfg.model_registry
assert REGISTRY, "set model_registry: `fsd init --set model_registry=abfss://…/model_registry`"
MODEL_NAME = "crop-rf"

# Test geometries, committed next to this notebook (notebooks/shapefiles/ -- see its NOTICE).
SHAPEFILES = NOTEBOOKS / "shapefiles"

# RESUMING A RUN. fsd skips work whose OUTPUT is already on disk, and it recognises that
# output by its PATH -- so re-running this notebook cheaply means pointing it back at the
# same paths. RESUME_RUN is how you do that.
#
# RESUME_RUN pins ROOT, where the IMAGERY + catalog live:
#     RESUME_RUN = None            -> new archive; the download leg re-downloads
#     RESUME_RUN = "demo-2026..."  -> re-enter that archive; [download] skips
RESUME_RUN = 'demo-20260824T121435Z'   # <- paste a previous RUN here to re-enter it
RUN  = RESUME_RUN or (
    "demo-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
ROOT = f"{AZ_ROOT}/{RUN}"
print("RUN =", RUN, "(resumed)" if RESUME_RUN else "(fresh)")

# TRAIN_RUN names the folder holding the datacubes. It is OPTIONAL -- "train" is already
# the default -- and is spelled out here for two reasons: it puts the cube folder on screen
# next to ROOT, and setting it to another run's folder is how you adopt cubes that already
# live somewhere else.
TRAIN_RUN = "train"
print("cube folder =", f"{ROOT}/runs/{TRAIN_RUN}")

runner_kwargs = dict(
    cluster = cfg.cluster,
    environment = f"{AZ_ENV_NAME}:{AZ_ENV_VERSION}",
    root = ROOT,
    identity_client_id = cfg.uami_client_id,   # the NODES' managed identity, not your login
    subscription_id = cfg.subscription_id,
    resource_group_name = cfg.resource_group,
    workspace_name = cfg.workspace,
    poll_interval_seconds = 10,
)

In [ ]:
print('ROOT =', ROOT)

In [ ]:
fields = gpd.read_file(SHAPEFILES / "AT_2018_TRAIN.geojson")   # 900 fields, cols: fid, crop

print('shape:', fields.shape, '\n')
print('columns:', fields.columns.tolist(), '\n')
print(fields['crop'].value_counts(), '\n')
fields.head()

## 1. Create training data (currently only Sentinel-2-L2A)

For a given:
- polygons with labels (labels are optional)
- `startdate`, `enddate`, `mosaic_days`
- `bands`
- `scl_mask_classes` # too hard-coded for sentinel-2-l2a
- source
- `export_folderpath`

The function creates arrays containing pixel-wise timeseries of bands and downloads to `export_folderpath`:
- data.npy            : Array containing timeseries of the bands
- coords.npy          : Array containing the lat-lon in EPSG:4326 of the pixels
- ids.npy             : Array containing the `id` of each entry in data.npy to map back to the polygons file
- labels.npy          : Array containing the label to each entry in data.npy
- metadata.pickle.npy : Contains information of the order of the bands in data.npy and more helper informations.

- features.npy        : After `"median_per_id"` aggregation
- feature_ids.npy
- feature_labels.npy

There are safe-guards put in place to prevent repeated works but needs further work.

If data is already downloaded:
```
  [plan] target: ./demo_training_data arrays -> CURRENT (stamp matches this request)
  [fetch] export -> ./demo_training_data | 8 files, 18.2 MB
```

Otherwise the plan prints before any work starts, then each leg reports its own shortfall:
```
  [plan] target: ./demo_training_data arrays -> STALE (no stamp)
  [plan]   flatten: 900 cubes required
  [plan] will run: build -> flatten -> land
  [plan]   build: 860 present, 40 missing, 0 known-empty -> will build 40
  [download] 0 of 828 assets missing; nothing to download
  [setup]    40/40 shapes (100%) | ...        <- only the shortfall, never all 900
  [aml]      run_id=... run_root=abfss://.../runs/...   <- before any job submits
  [build]    40 of 900 cubes missing; dispatching 40
  [flatten]  arrays match the current 900 cubes; skipping
```

**Features:**
- CHANGING A PARAMETER CHANGES THE PATHS. The cube folder name carries a short digest of (bands, mosaic scheme, scl_mask_classes), so a request differing in any of them writes somewhere else and does not adopt the old cubes -- expect a full fan-out after such a change. The old cubes are left alone on blob, not deleted.

- FIELDS WITH NO IMAGERY are recorded once in {ROOT}/runs/{TRAIN_RUN}/_manifest.json and reported as "known-empty" from then on, so re-runs do not chase them forever. Changing the window or the parameters clears that record; re-ingesting the ARCHIVE under an unchanged request does not -- use overwrite= for that.
 
- overwrite= forces a leg past its skip:
    - False (default) skip what's done | "datacubes" rebuild cubes (and re-flatten)
    - "flatten" keep cubes, redo the flatten | True both  
    - "datacubes" and True also refresh every per-field catalog slice and clear the known-empty record -- that is the escape hatch after re-ingesting imagery.


**TODO:**
1. Make things more abstract to make addition of a different satellite source more simpler. [CRITICAL]
2. Improve the traceback to decide what steps to execute.

In [ ]:
td = fsd.create_training_data(
    label_polygons = fields,                       # in-memory gdf -> auto-staged to blob
    catalog_filepath = f"{ROOT}/imagery/catalog.parquet",  # {ROOT}/imagery is the folder where the images would be downloaded to.
    startdate = datetime.datetime(2018, 4, 1),
    enddate = datetime.datetime(2018, 9, 30),
    mosaic_days = 20,
    bands = ["B04", "B08", "B8A", "SCL"],
    id_col = "fid", label_col = "crop",
    scl_mask_classes = [0, 1, 3, 7, 8, 9, 10],
    export_folderpath = "./demo_training_data",    # LOCAL landing dir for the compact array
    aggregate = "median_per_id",                   # one row per field, not per pixel
    source = "mpc", download = True,
    overwrite = False,                             # skip whatever is already done
    max_tiles = 250, max_cloudcover = 70,
    runner = "aml", runner_kwargs = runner_kwargs,
    run_folderpath = f"{ROOT}/runs/{TRAIN_RUN}",   # STABLE across runs -- see TRAIN_RUN above
)

# 13m 39.8s

## 2. Model training

The training data that was downloaded currently contains raw bands.

1. create the features we want to train our model with. In our case: `['NDVI', 'SAVI']`
2. train a random-forest model.

### 2.1 Create features

In [ ]:
import numpy as np
from fsd.bands import modify
from fsd.model.features import apply_features   # the one place feature_sequence is applied

In [ ]:
d = td.load()
raw, ids, y = d["features"], d["feature_ids"], np.asarray(d["feature_labels"])  # (900,10,3)
band_indices = {b: i for i, b in enumerate(td.bands)}                           # B04,B08,B8A

# create features NDVI and SAVI
SEQ = [
    (modify.mask_invalid_and_interpolate, {}),
    (modify.compute_bands, dict(bands_to_compute=["NDVI", "SAVI"])),
    (modify.remove_bands, dict(bands_to_remove=["B04", "B08", "B8A"])),
]

feats5d, feat_bi = apply_features(modify.expand_flattened(raw.astype(float)),
                                  dict(band_indices), feature_sequence=SEQ)
feats = np.squeeze(feats5d, axis=(2, 3))         # (900, 10, n_features)

print(feats.shape, sorted(feat_bi, key=feat_bi.get))

### 2.2 Train

In [ ]:
import joblib, os
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score

In [ ]:
X = feats.reshape(len(feats), -1)
keep = ~np.isnan(X).any(axis=1)
X, yk = X[keep], y[keep]
le = LabelEncoder(); yy = le.fit_transform(yk)

clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
print("field-wise CV:", cross_val_score(clf, X, yy, cv=5).mean())   # rows are fields → honest
clf.fit(X, yy)

outdir = "./demo_model"
os.makedirs(outdir, exist_ok=True)
model_fp = os.path.join(outdir, "rf.joblib")
joblib.dump((clf, le), model_fp)

## 3. Deploying the model

**"Deploy" is five steps, not one.** Each is a gate: it either passes, or it tells you which of
the five is wrong. That is the whole reason they are separate calls rather than one button.

| | step | what it proves | where it runs |
|---|---|---|---|
| 3.1 | write the model adapter | nothing yet — this is code you write | — |
| 3.2 | `fsd.verify_adapter` | your adapter runs on a **real** datacube, and the output looks right in QGIS | one AML node, then your machine |
| 3.3 | `bundle.save` | model + code + requirements travel as **one unit** | your machine |
| 3.4 | `fsd.model.verify_image` | the inference **image** can load and run that bundle | one AML node |
| 3.5 | `fsd.deploy` | the verified bundle gets a **name** in the registry | your machine + blob |

**A prerequisite that is not in this notebook.** Step 3.4 needs the inference image to already
exist — `fsd-infer-sklearn`, built and registered by
[`00_build_images.ipynb`](./00_build_images.ipynb). It is generic per dependency *family*, so
you build it once and every sklearn model you ever train reuses it. Retraining never touches it.

**One ordering rule.** 3.3 -> 3.4 -> 3.5 run in that order, and you must not re-run 3.3 in
between. `deploy` accepts 3.4's verdict (`verified=vres`) only while the bundle content that
verdict fingerprinted still matches what you are deploying. Re-bundling invalidates the proof,
and `deploy` refuses rather than registering something the image was never run against.


### 3.1 Creating a model adapter

The adapter is how FSD's inference pipeline uses the model

For this example the adapter is written to ./demo_model/my_adapter.py

```python
from fsd.bands import modify
from fsd.model.adapter import BaseModelAdapter

class CropRF(BaseModelAdapter):
    required_bands = ["B04", "B08"]        # raw bands SEQ consumes
    n_timestamps = 0                       # set per-instance at bundle time
    output_dtype = "uint8"
    output_nodata = 255
    output_band_names = ["crop_class"]
    feature_sequence = [
        (modify.mask_invalid_and_interpolate, {}),
        (modify.compute_bands, dict(bands_to_compute=["NDVI", "SAVI"])),
        (modify.remove_bands, dict(bands_to_remove=["B04", "B08", "B8A"])),
    ]

    def load(self):
        import joblib
        self.clf, self.le = joblib.load(self.artifacts["model"])

    def predict(self, X_chunk):
        return self.clf.predict(X_chunk).astype("uint8")
```

The next cell, `fsd.verify_adapter`, runs this adapter on one real datacube before you trust
it at scale.

**The adapter must REPLAY the features you trained on.** `feature_sequence` above is the same
three operations as `SEQ` in section 2.1 — and **nothing in fsd checks that**. If the two drift
apart, every inference node computes different inputs than training used, and you get confident
nonsense with no error anywhere: not at bundling, not at `verify_adapter` (which only proves the
adapter *runs*), not at inference. The next cell is that check, done by hand, because it is the
one mistake in this notebook that fails silently.


In [ ]:
# The one check fsd cannot do for you (see above). SEQ is what section 2.1 trained on;
# CropRF.feature_sequence is what every inference node replays. They must be identical.
import sys; sys.path.insert(0, "./demo_model")
from my_adapter import CropRF

assert CropRF.feature_sequence == SEQ, (
    "the adapter's feature_sequence has drifted from the SEQ you trained on:\n"
    f"  trained on : {SEQ}\n"
    f"  adapter    : {CropRF.feature_sequence}"
)
print("features match:", [fn.__name__ for fn, _ in SEQ])


### 3.2 Verify the adapter

Verify the adapter by running the model on one datacube.

`fsd.verify_adapter` builds **one** ~5km grid cell datacube on AML, downloads it, and runs the model via the adapter locally.
The output of the adapter is `output.tif`, which is to be opened on QGIS to verify if model worked as expected.

We are not checking whether the image - which was created in 00_build_images.ipynb - works against the model.

We are just checking if the adapter, as written, works - because the user needs write the code to interact with the datacube themselves.

**Notes on the call below:**

- It takes the **live adapter**, not `bundle_dir` — so the run exercises **bundling itself**
  (`verify_adapter` auto-saves to a temp bundle, exactly as `run_inference` would). That is why
  this sits *before* the bundling cell rather than after it.
- `cell=None` picks **deterministically** — the cell with the largest in-window catalog coverage,
  so a failure is your adapter's fault and not an empty cell's — and prints which cell and why.
  Paste that id back as `cell="..."` to pin it. `cell="random"` samples instead, and prints its
  pick so it can be pinned too. Cost note: the deterministic pick filters the catalog once per
  grid cell on the driver (299 passes for `AT_ROI`), so give it a moment.
- `bands` must be the bands **`run_inference` will use** (`B04`/`B08`/`SCL`), not the model's
  `required_bands` — the point is to build the cube the fan-out will actually hand `predict`.
- `runner_kwargs` is the **training** one (`fsd-aml-env`): only the *cube build* goes to AML. The
  inference runs right here, which is what makes it debuggable — drop a breakpoint in `predict`.
- `export_folderpath` is keyed to `RUN` on purpose. The resume check covers the *request*
  (roi/window/bands/cell), not which archive it came from — so one folder shared across two
  different `ROOT`s would reuse the older run's cube. Under `RESUME_RUN` it resumes correctly and
  skips the build entirely; that is the tight loop worth having.


In [ ]:
# demo_model/ holds exactly two things: the adapter source and the trained model.
# No Dockerfile, no wheel -- those belong to the IMAGE, not to the model (see 00_build_images).
# CropRF itself was imported in 3.1, by the feature-parity check.
from fsd.model import bundle


In [ ]:
adapter = CropRF()
adapter.n_timestamps = fsd.compute_n_timestamps(
    datetime.datetime(2018, 4, 1), datetime.datetime(2018, 9, 30), 20)   # 10
adapter.artifacts = {"model": model_fp}

# One cell, built on AML, landed here, run through the real inference unit.
# First call: ~ one AML job + a transfer. Later calls with the SAME request: no job at all,
# straight to inference -- so iterating on my_adapter.py costs seconds, not a cluster.
report = fsd.verify_adapter(
    adapter,                                          # the LIVE adapter -> auto-bundled
    roi              = str(SHAPEFILES / "AT_ROI.geojson"),
    catalog_filepath = f"{ROOT}/imagery/catalog.parquet",
    startdate        = datetime.datetime(2018, 4, 1),
    enddate          = datetime.datetime(2018, 9, 30),
    mosaic_days      = 20,
    bands            = ["B04", "B08", "SCL"],         # what run_inference builds, not required_bands
    scl_mask_classes = [0, 1, 3, 7, 8, 9, 10],
    cell             = None,                          # deterministic pick; prints the id + why
    export_folderpath = f"./demo_verify_adapter/{RUN}",
    storage = "azure", runner = "aml", runner_kwargs = runner_kwargs,
)

# pass: False is a finding ABOUT THE ADAPTER (a T mismatch, a wrong output dtype) and comes back
# in the dict; only a bad CALL raises. Either way the verdict is on disk as _result.json.
assert report["pass"], report["error"]

m = report["metrics"]
print("cell           ", m["cell"])
print("cube shape     ", m["cube_shape"], "T =", m["cube_t"], "(adapter wants", m["adapter_n_timestamps"], ")")
print("bands in cube  ", m["cube_bands"])
print("after features ", m["post_feature_sequence_bands"], "| required:", m["required_bands"])
print("output         ", m["output_dtype"], "range", (m["output_value_min"], m["output_value_max"]),
      "| nodata frac", m["output_nodata_fraction"])
print()
print("OPEN THESE IN QGIS -- the verdict above is assistive, the raster is the deliverable:")
print("  ", m["output_filepath"])
print("  ", m["grids_filepath"])

# 2m 51.2s - 6m 34.6s (node startup sometimes take 5 minutes)
# 3.3s (pre-ready datacube)

### 3.3 Bundling the model

A model bundle is three things:
1. The model binary (rf.joblib) - DONE
2. The code to run the model (my_adapter.py) - DONE
3. The list of requirements to run the model's code - what we do NOW.

`bundle.save` will put the model binary and the model adapter in a "bundle" directory and creates a bundle.json file containing the requirements to run the model along with other model requirements - bands, operations, data shape - which is how FSD knows whether the model would run and what it needs to run the model before we launch a large scale inference run.

In [ ]:
bundle_dir = bundle.save(
    adapter = adapter,
    artifacts = {"model": model_fp},
    dst = "./demo_bundle",
    code = ["./demo_model/my_adapter.py"],
    requirements = ["scikit-learn>=1.5", "joblib"],
)

spec = bundle.read_spec(bundle_dir)
print(spec["adapter"])          # my_adapter:CropRF
print(spec["fsd_bundle_version"], spec["code"])   # 2  {'root': 'code', 'files': ['my_adapter.py']}

### 3.4 Verify inference image

Verifying whether the image chosen works for the model.

If this fails then the fix is either making corrections in the adapter, or creating a new image - latter would involve re-running "00_build_images.ipynb" with custom configuration.

In [ ]:
from fsd.model import verify_image

vres = verify_image(
    bundle_dir,
    environment    = f"{AZ_INFER_ENV_NAME}:{AZ_INFER_ENV_VERSION}",
    runner         = "aml",
    runner_kwargs  = runner_kwargs,
    image_ref      = _infer.registry_ref,   # [OPTIONAL] spec 56 D8: the same staleness gate,
    registry       = IMAGE_REGISTRY,        # read from the image registry instead of a wheel.
                                            # NOT _infer.ref -- that is AML's version number;
                                            # the registry numbers definitions separately.
)
assert vres["pass"], vres               # don't spend a 299-cell fan-out on an unverified image
vres["metrics"]

# 3m 24.4s

### 3.5 Deploy the model to the registry

**This is where a verified model gets a NAME.** `fsd.deploy` publishes `./demo_bundle` into
`REGISTRY` — a folder on **blob**, not on this laptop — and hands back a reference, `crop-rf:1`,
that the inference cell below accepts in place of a path.

`deploy` refuses to register a model that has not been *proven to run on the image it names*. That proof is the `verify_image` call directly
above, so this is the first point in the notebook where deploying is even possible.

Passing `verified=vres` reuses that result instead of paying a second ~3½-minute node — but only
if it genuinely speaks to this model. The match is on the content fingerprint `verify_image` took
**at the moment it verified**, so if you re-run the bundling cell after verifying, this call is
**refused as stale** rather than registering something the image never saw. Re-run `verify_image`,
then this.

**What a name buys you:**

| | a path | a name |
|---|---|---|
| the inference cell says | `./demo_bundle`, a folder only this laptop has | `crop-rf@demo-eurocrops-at`, resolvable from anywhere |
| retraining means | edit the inference cell | re-run this cell; the alias moves |
| "which model made this output?" | check when the folder last changed | `v3`, printed in the run log |

`demo-eurocrops-at` is an **alias** — a movable pointer to a version. Versions themselves are immutable:
re-running this cell with unchanged model content gives you the **same version back** and creates
no new version folder. Change the model and you get `v2`; `v1`'s bytes are never touched, so an
output from last week can still be reproduced.

**Why the registry is on blob now.** Until 2026-08-25 this cell needed a local folder, and this
notebook said so in three places. Two changes removed that: the publish protocol was proven
against a real `abfss://` account, and the local run path was fixed — a blob-resolved bundle is
fetched to scratch before loading, because `sys.path` cannot hold a URL and an `abfss://` entry
there is simply inert. `storage="azure"` below is what forbids the anonymous fallback, so a
credential problem surfaces as a refusal instead of a confusing 404.


In [ ]:
MODEL_ALIAS = "demo-eurocrops-at"

MODEL_REF = fsd.deploy(
    bundle_dir,
    name        = MODEL_NAME,               # "crop-rf" -- no "/", ":" or "@" (a ref must parse)
    registry    = REGISTRY,                 # on BLOB -- see the config cell
    storage     = "azure",                  # forbid the anonymous fallback for that write
    environment = f"{AZ_INFER_ENV_NAME}:{AZ_INFER_ENV_VERSION}",   # SAME string verify_image got
    verified    = vres,                     # reuse the gate above; stale/mismatched -> refused
    alias       = MODEL_ALIAS,      # publish + repoint in one call
)
print(MODEL_REF)                            # crop-rf:1  <- a version PIN, always exact

# What landed. Nothing the registry writes contains an absolute path, so the whole folder
# can be moved or copied elsewhere and every name still resolves to the same version.
from fsd.model import registry as _registry

resolved = _registry.resolve(f"{MODEL_NAME}@{MODEL_ALIAS}", REGISTRY)
print(resolved)                              # Resolved(name=..., version=1, path=...)

rec = _registry.read_deploy_record(resolved.path)   # _deploy.json -- the bundle<->image binding
print("environment :", rec["environment"])   # what this version was PROVEN to run on
print("digest      :", rec["digest"])        # what content that proof covered
print("deployed_at :", rec["deployed_at"])

## 4. Run inference

Step 5 of the map at the top, and the only step that is two calls: `run_inference` reads a
catalog and never fetches imagery itself, so a new ROI needs `fsd.download` first.


### 4.1 Download the imagery inference will read


In [ ]:
# For MPC, discovery runs here on your machine and the result is diffed against the catalog
# BEFORE anything is dispatched, so a no-op download costs seconds rather than a cold start:
#   [download] 0 of 828 assets missing; nothing to download   <- submits no job at all
#   [download] 41 of 828 assets missing; dispatching 41       <- ships only the 41
# max_tiles is counted against that shortfall, not against every discovered tile.
# CDSE works differently: it submits one whole-ROI job and discovers on the node.

fsd.download(roi = str(SHAPEFILES / "AT_ROI.geojson"),
             dst_folderpath = f"{ROOT}/imagery",
             startdate = datetime.datetime(2018, 4, 1),
             enddate = datetime.datetime(2018, 9, 30),
             bands = ["B04", "B08", "B8A", "SCL"],
             source = "mpc",
             max_tiles = 250, max_cloudcover = 70,
             runner = "aml", runner_kwargs = runner_kwargs)

# 5.3s

### 4.2 Run inference off the deployed NAME


In [ ]:
infer_kwargs = dict(runner_kwargs)
infer_kwargs["environment"] = f"{AZ_INFER_ENV_NAME}:{AZ_INFER_ENV_VERSION}"

In [ ]:
# `output_folderpath` IS the identity of a run. What still needs doing is worked out from the
# cell list cached inside that folder -- so if you point a DIFFERENT roi at a folder that
# already has one, fsd raises rather than quietly inferring the old roi's cells.
#   -> switching the roi below REQUIRES switching output_folderpath too. Both lines, together.
#
# Expect progress from the merge leg as well as the fan-out:
#   [collect] 0/299 candidates (0%) | elapsed 0s
#   [merge]   137/299 inputs (46%) | 0.3 inputs/s | elapsed 452s | eta 540s
#   [merge]   299/299 inputs reprojected (100%) | ...
#   [merge]   merging 299 inputs into the mosaic (reads pixels; no per-input progress)
#
# One thing to know about re-running: this leg decides per cell whether the output already
# exists ON THE NODE, after the job has started. So a 95%-complete re-run still starts ~299
# tasks to find out it only needs ~15. overwrite=False below is that per-output skip.

result = fsd.run_inference(
    # A NAME, resolved against the registry on blob, instead of a path to a folder.
    # `@demo-eurocrops-at` is an ALIAS -- "whatever is current" -- so this cell does not change
    # when you retrain; re-running the deploy cell above repoints it. That does make the run's
    # meaning depend on WHEN you ran it, so resolution says out loud what it picked, before any
    # node starts:
    #     [model] crop-rf@demo-eurocrops-at -> v1 (verified against fsd-infer-sklearn:6)
    # Pin a version instead -- model = f"{MODEL_NAME}:1" -- when you want a run that cannot move
    # under you. A plain bundle path still works too: model = bundle_dir.
    model    = MODEL_REF,
    registry = REGISTRY,
    # output_folderpath = f"{ROOT}/model_outputs/s2grid=476da24",
    # roi = str(SHAPEFILES / "s2grid=476da24.geojson"),
    output_folderpath = f"{ROOT}/model_outputs/AT_ROI",
    roi = str(SHAPEFILES / "AT_ROI.geojson"),
    catalog_filepath = f"{ROOT}/imagery/catalog.parquet",
    startdate = datetime.datetime(2018, 4, 1),
    enddate = datetime.datetime(2018, 9, 30),
    mosaic_days = 20,
    bands = ["B04", "B08", "SCL"],
    scl_mask_classes = [0, 1, 3, 7, 8, 9, 10],
    merge = "reproject", overwrite = False,
    storage = "azure", runner = "aml", runner_kwargs = infer_kwargs,
)
print(len(result.output_filepaths), result.merged_filepath, result.stac_catalog_filepath)

# str(SHAPEFILES / "s2grid=476da24.geojson") ->   1 cell   -- a few minutes
# str(SHAPEFILES / "AT_ROI.geojson")         -> 299 cells  -- 23m 23.5s

## Things to know before you rely on this

None of these stop the notebook running. They are the places where fsd will behave in a way you
would not guess, so they are worth reading once.

**One leg of the blob registry is on its first real run.** Publishing, resolving and running off
an `abfss://` registry all work. But with `runner="aml"` the bundle is copied registry -> run-root
**blob to blob**, and every earlier real run did that copy from a *local* bundle. It works on
`memory://`; if staging behaves oddly, that is the leg to suspect first.

**Do not pass `storage="azure"` on the LOCAL inference path.** It is refused there
([#90](https://github.com/nikhilsrajan/fsd/issues/90)) — the seam gate treats `storage=` and
`registry=` as one axis when they are independent. ROI + `runner="aml"`, which is what this
notebook uses, is the shape where it is both allowed and doing real work. And on any local run,
keep `output_folderpath` local too ([#91](https://github.com/nikhilsrajan/fsd/issues/91)).

**Downloads and datacube writes are not atomic.** If a transfer or a cube write is interrupted,
a partial file is left under its final name, and the "is it already there?" check on the next run
counts it as done — so the run that would have fixed it never happens. If a run died midway and
something downstream looks wrong, re-run that leg with `overwrite=` rather than trusting the skip.

**Re-running inference is cheaper than it looks, but not free.** The fan-out decides per cell
whether the output already exists *on the node*, after the container has started. A nearly
complete re-run therefore still starts a task per cell to discover it has little to do.

**The final merge runs on your machine, not on the cluster.** It pulls every per-cell GeoTIFF
back over the network, which is the slowest part of a large ROI — roughly 1000 s for 300 cells
over a VPN. It reports progress, but the cost is real; budget for it.

**A `verify_adapter` cube does not record which archive it came from.** The resume check covers
your request (roi, window, bands, cell), not `catalog_filepath` — so one `export_folderpath`
shared across two different `ROOT`s would reuse the older archive's cube. That is why
`export_folderpath` above is keyed to `RUN`.

**Nothing warns you if the image you run on differs from the one you deployed against.** The
`[model] … (verified against …)` line tells you which image the model was *proven* on; comparing
that to the image the fan-out actually used is currently your job. `verify_image` is what
actually establishes a model/image pairing works, so run it whenever either side changes.

**Only Sentinel-2 L2A, and only MPC gets the cheap no-op download.** CDSE submits one whole-ROI
job and discovers on the node, so a CDSE re-run does not short-circuit the way an MPC one does.
